# E2：多目标优化 —— 动力 vs 成本权衡（pymoo）

## 实验目标
- 理解 Pareto 前沿与多目标权衡
- 用 NSGA-II 做双目标优化

## 通过判据
```python
assert len(pareto_front) >= 5
```


In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np
print('✓ pymoo 导入成功')


In [ ]:
class EVDesign(Problem):
    """电动车设计：加速性能 vs 购车成本"""
    """权衡：大功率加速快但贵，小功率便宜但慢"""
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=0,
                         xl=np.array([50., 1000.]), xu=np.array([200., 2500.]))
    def _evaluate(self, x, out, *args, **kwargs):
        power, weight = x[:, 0], x[:, 1]
        t_0100 = 2.5 * weight / power  # 加速时间
        cost = 15000 + 80 * power + 5 * weight  # 成本
        out['F'] = np.column_stack([t_0100, cost])

problem = EVDesign()
res = minimize(problem, NSGA2(pop_size=60), ('n_gen', 50),
               verbose=False, seed=42)
pareto = res.F
print(f'Pareto 解数: {len(pareto)}')
assert len(pareto) >= 5, 'Pareto 解太少!'
print('✅ 评测通过！')


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(pareto[:, 0], pareto[:, 1], c='blue', alpha=0.6)
plt.xlabel('0-100km/h Time (s)')
plt.ylabel('Cost (CNY)')
plt.title('Pareto Front: Performance vs Cost')
plt.grid(True, alpha=0.3)
plt.show()


## 🏆 挑战任务
1. 加第三个目标：百公里电耗，做三目标优化。
2. 从 Pareto 前沿选一个折中解，用 FASTSim 验证真实能耗。
